In [9]:
import pandas as pd
import chemsource
import os
import sys
import time
import asyncio
import copy
import openai

sys.path.append(os.path.abspath("../src"))
from harmonization import (
    harmonize_automated_classification,
    harmonize_manual_classification,
)


In [ ]:
pd.read_parquet("/Users/prajitrajkumar/Downloads/MSV000098263_envedams_harmonized.parquet")

,FEATURE_ID,M/Z,RT,ION_MOBILITY,CCS,ADDUCT,MS/MS_ASSIGNED,MS/MS_MZS,MS/MS_INTENSITIES,P3-A7_B,...,P2_F8_B,P3-D10_C,P3-B4_B,P1-E7_C,P2_H4_C,P2_G12_A,P1-B2_C,P1-F1_C,P1-G4_B,P2_G1_B
0,0,547.318032,5.125636,0.797072,163.646736,[M+H]+,False,[],[],681.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,10000,353.239519,5.916804,0.896676,186.542429,[M+H]+,False,[],[],0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10001,375.125556,1.734383,0.888188,184.380160,[M+H-CH3]+,False,[],[],0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,10002,247.174722,8.925751,0.461164,97.440322,[M+H]+,False,[],[],0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10003,421.183041,8.272833,1.313450,271.623021,[M+H]+,False,[],[],0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24165,9998,384.168142,1.685255,0.950135,197.078561,[M+H]+,True,"[37.00043869, 38.00955963, 39.01615906, 41.032...","[849.0, 155.0, 1122.0, 5918.0, 1634.0, 454.0, ...",0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
24166,9999,429.211155,2.258600,1.066151,220.352684,[M+H]+,False,[],[],0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
24167,999,611.151148,2.694633,1.180940,241.841117,[M+H]+,False,[],[],0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
24168,99,469.261563,2.374918,1.052156,216.890968,[M+H]+,False,[],[],0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


: 

In [ ]:
classified_drug_library_data_path = "../data/drug_library/validation_data_classified_all_3_methods.csv"

harmonized_automated = harmonize_automated_classification(classified_drug_library_data_path)
harmonized_manual = harmonize_manual_classification(classified_drug_library_data_path)


harmonized_automated_non_medical = harmonized_automated[harmonized_automated['GPT_RAG'].apply(lambda x: 'MEDICAL' not in x and "INFO" not in x)]
harmonized_automated_medical = harmonized_automated[harmonized_automated['GPT_RAG'].apply(lambda x: 'MEDICAL' in x and "INFO" not in x)]
harmonized_automated_info = harmonized_automated[harmonized_automated['GPT_RAG'].apply(lambda x: 'INFO' in x)]

drug_library_text = pd.read_csv(classified_drug_library_data_path)
drug_library_text["FEATURE_ID"] = drug_library_text.index
drug_library_text = drug_library_text[["FEATURE_ID", 
    "name_used", 
    "text"]].rename(columns={
    "name_used": "NAME",
    "text": "TEXT"})

: 

In [ ]:
harmonized_automated[drug_library_text["TEXT"].apply(lambda x: True if type(x) is not str else False).to_numpy()]

,FEATURE_ID,SOURCE,DEEPSEEK_RAG,GPT_NO_RAG,GPT_RAG,SEARCH_GPT
52,52,NaN,[INFO],[MEDICAL],[INFO],[INFO]
107,107,NaN,[INFO],[MEDICAL],[INFO],[MEDICAL]
145,145,NaN,[INFO],[MEDICAL],[INFO],[INFO]
312,312,NaN,[INFO],[INFO],[INFO],[INFO]
543,543,NaN,[INFO],[INFO],[INFO],[INFO]
692,692,NaN,[INFO],[MEDICAL],[INFO],[INFO]
734,734,NaN,[INFO],[MEDICAL],[INFO],[INFO]
738,738,NaN,[INFO],[INFO],[INFO],[INFO]
820,820,NaN,[INFO],[INFO],[INFO],[INFO]
1314,1314,NaN,[INFO],[MEDICAL],[INFO],[INFO]


: 

In [ ]:
openai_api_key = open("../secrets/openai_api_key.txt").read().strip()
ncbi_api_key = open("../secrets/ncbi_api_key.txt").read().strip()

chem_gpt_4o = chemsource.ChemSource(
    model_api_key=openai_api_key,
    ncbi_key=ncbi_api_key,
    model="gpt-4o",
    clean_output=True,
    allowed_categories=["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
)

prompt_gpt_4o = "You are a helpful scientist that will classify the provided compound \
COMPOUND_NAME using only the information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"


chem_gpt_4o.prompt = prompt_gpt_4o



: 

In [ ]:
def classification_to_bits(classification):
    categories = ["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
    bits = ["1" if category in classification else "0" for category in categories]

    if len(set(classification) - set(categories)) > 0:
        unknown_categories = ",".join(set(classification) - set(categories))
        return "".join(bits)+f"_UNKNOWN({unknown_categories})"
    return "".join(bits)



async def async_run(data_list, model_instance):
    # Get the current running event loop instead of get_event_loop()
    loop = asyncio.get_running_loop()
    # Create tasks for each row - now passing tuples of (name, text)
    futures = [loop.run_in_executor(None, model_instance.classify, name, text) for name, text in data_list]
    # Gather all results
    result = await asyncio.gather(*futures, return_exceptions=True)
    return result



: 

In [ ]:
drug_library_text_random_sample = drug_library_text.sample(n=100,
                                                           random_state=67)
drug_library_text_random_sample_data = list(drug_library_text_random_sample[["NAME", "TEXT"]].itertuples(index=False, name=None))
drug_library_text_random_sample_no_text = copy.deepcopy(drug_library_text_random_sample)
drug_library_text_random_sample_no_text[["TEXT"]] = " "
drug_library_text_random_sample_no_text_data = list(drug_library_text_random_sample_no_text[["NAME", "TEXT"]].itertuples(index=False, name=None))

: 

In [ ]:
drug_library_total_tokens = drug_library_text.apply(lambda row: len(row["TEXT"]) if type(row["TEXT"]) is str else 0, axis=1).sum()
drug_library_random_sample_tokens = drug_library_text_random_sample.apply(lambda row: len(row["TEXT"]), axis=1).sum()

token_ratio = drug_library_total_tokens / drug_library_random_sample_tokens

: 

In [ ]:
start_time = time.perf_counter()

classifications_gpt4o = await async_run(drug_library_text_random_sample_data, chem_gpt_4o)

end_time = time.perf_counter()

: 

In [ ]:
time_elapsed_gpt_4o = end_time - start_time
print(f"Time elapsed for 100 compounds (seconds): {time_elapsed_gpt_4o}")

total_time_estimate_gpt_4o = time_elapsed_gpt_4o * token_ratio
print(f"Total time estimate (seconds, GPT-4o): {total_time_estimate_gpt_4o}")

Time elapsed for 100 compounds (seconds): 7.405757500004256
Total time estimate (seconds, GPT-4o): 331.41953206545566


: 

In [ ]:
classifications_gpt4o.append([time_elapsed_gpt_4o])

: 

In [ ]:
pd.DataFrame(classifications_gpt4o).to_csv("../data/drug_library/measure_time_sample_results/gpt_4o_time_classification_results.csv")

: 

In [ ]:
start_time_gpt_4o_linear = time.perf_counter()

classifications_gpt_4o_linear = []
for index, row in drug_library_text_random_sample.iterrows():
    compound_name = row["NAME"]
    compound_text = row["TEXT"]
    classification = chem_gpt_4o.classify(compound_name, compound_text)
    classifications_gpt_4o_linear.append(classification)

end_time_gpt_4o_linear = time.perf_counter()

: 

In [ ]:
time_elapsed_gpt_4o_linear = end_time_gpt_4o_linear - start_time_gpt_4o_linear
print(f"Time elapsed for 100 compounds (seconds): {time_elapsed_gpt_4o_linear}")

total_time_estimate_gpt_4o_linear= time_elapsed_gpt_4o_linear * token_ratio
print(f"Total time estimate (seconds, GPT-4o, linear): {total_time_estimate_gpt_4o_linear}")

Time elapsed for 100 compounds (seconds): 68.59615566600405
Total time estimate (seconds, GPT-4o, linear): 3069.788041574836


: 

In [ ]:
with open("../data/drug_library/measure_time_sample_results/gpt_4o_linear_time_results.txt", "a") as f:
    f.write(str(time_elapsed_gpt_4o_linear))

: 

In [ ]:
chem_gpt_4_1 = chemsource.ChemSource(
    model_api_key=openai_api_key,
    ncbi_key=ncbi_api_key,
    model="gpt-4.1",
    clean_output=True,
    allowed_categories=["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
)

prompt_gpt_4_1 = "You are a helpful scientist that will classify the provided compound \
COMPOUND_NAME as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification.\n"


chem_gpt_4_1.prompt = prompt_gpt_4_1



: 

In [ ]:
start_time_gpt_4_1 = time.perf_counter()

classifications_gpt_4_1 = await async_run(drug_library_text_random_sample_no_text_data, chem_gpt_4_1)

end_time_gpt_4_1 = time.perf_counter()

: 

In [ ]:
time_elapsed_gpt_4_1 = end_time_gpt_4_1 - start_time_gpt_4_1
print(f"Time elapsed for 100 compounds (seconds): {time_elapsed_gpt_4_1}")

total_time_estimate_gpt_4_1 = time_elapsed_gpt_4_1 * token_ratio
print(f"Total time estimate (seconds, GPT-4.1): {total_time_estimate_gpt_4_1}")

Time elapsed for 100 compounds (seconds): 4.648809292004444
Total time estimate (seconds, GPT-4.1): 208.0416757120078


: 

In [ ]:
classifications_gpt_4_1.append([time_elapsed_gpt_4_1])

: 

In [ ]:
pd.DataFrame(classifications_gpt_4_1).to_csv("../data/drug_library/measure_time_sample_results/gpt_4_1_time_classification_results.csv")

: 

In [ ]:
start_time_gpt_4_1_linear = time.perf_counter()

classifications_gpt_4_1_linear = []
for index, row in drug_library_text_random_sample_no_text.iterrows():
    compound_name = row["NAME"]
    compound_text = row["TEXT"]
    classification = chem_gpt_4_1.classify(compound_name, compound_text)
    classifications_gpt_4_1_linear.append(classification)

end_time_gpt_4_1_linear = time.perf_counter()

: 

In [ ]:
time_elapsed_gpt_4_1_linear = end_time_gpt_4_1_linear - start_time_gpt_4_1_linear
print(f"Time elapsed for 100 compounds (seconds): {time_elapsed_gpt_4o_linear}")

total_time_estimate_gpt_4_1_linear= time_elapsed_gpt_4_1_linear * token_ratio
print(f"Total time estimate (seconds, GPT-4.1, linear): {total_time_estimate_gpt_4_1_linear}")

Time elapsed for 100 compounds (seconds): 68.59615566600405
Total time estimate (seconds, GPT-4.1, linear): 2499.604846034595


: 

In [ ]:
with open("../data/drug_library/measure_time_sample_results/gpt_4_1_linear_time_results.txt", "a") as f:
    f.write(str(time_elapsed_gpt_4_1_linear))

: 

In [ ]:
from typing import Optional, Any, Union, List
from spellchecker import SpellChecker
from openai import OpenAI

def custom_search_classify(name: str,
             input_text: Optional[str] = None, 
             api_key: Optional[str] = None, 
             baseprompt: Optional[str] = None,
             model: str = 'gpt-4.1', 
             temperature: float = 0,
             top_p: float = 0,
             max_length: int = 250000,
             clean_output: bool = False,
             explanation: bool = False,
             explanation_separator: str = "EXPLANATION_COMPLETE",
             output_explanation: bool = False,
             allowed_categories: Optional[List[str]] = None,
             custom_client: Optional[Any] = None,
             spell_checker: Optional[SpellChecker] = None) -> Union[str, List[str]]:
    """
    Classify a chemical compound using an AI language model.
    
    This function takes a chemical compound name and additional information,
    then uses an AI model to classify it into predefined categories.
    
    Args:
        name (str): The name of the chemical compound to classify.
        input_text (str, optional): Additional information about the compound.
        api_key (str, optional): API key for the language model service.
        baseprompt (str, optional): Base prompt template for classification.
        model (str, optional): Name of the language model to use. Defaults to 'gpt-4o'.
        temperature (float, optional): Temperature parameter for model creativity. Defaults to 0.
        top_p (float, optional): Top-p parameter for nucleus sampling. Defaults to 0.
        max_length (int, optional): Maximum length of the prompt in characters. Defaults to 250000.
        clean_output (bool, optional): Whether to clean and validate the output. Defaults to False.
        explanation (bool, optional): Whether to expect and extract explanations from the model response.
                                     Only used when clean_output=True. The model's response should contain
                                     an explanation followed by the separator, then the classification.
                                     Defaults to False.
        explanation_separator (str, optional): The delimiter string that separates the explanation from 
                                              the classification in the model's response. Only used when
                                              both clean_output=True and explanation=True.
                                              Defaults to "EXPLANATION_COMPLETE".
        output_explanation (bool, optional): Whether to return the explanation text alongside classification.
                                            When True, returns a tuple (classification_list, explanation_text).
                                            Only used when both explanation=True and clean_output=True.
                                            Defaults to False.
        allowed_categories (List[str], optional): List of allowed categories for filtering output.
        custom_client (Any, optional): Custom OpenAI client instance.
        spell_checker (SpellChecker, optional): Spell checker instance for output correction.
    
    Returns:
        Union[str, List[str], Tuple[List[str], str]]: 
            - If clean_output=False: Raw model output string
            - If clean_output=True and output_explanation=False: List of categories
            - If clean_output=True, explanation=True, and output_explanation=True: 
              Tuple of (category_list, explanation_text)
    
    Raises:
        ValueError: If clean_output is True but allowed_categories is None, or if
                   output_explanation=True but explanation=False.
        IndexError: If explanation=True but the explanation_separator is not found in the response.
        
    Example:
        >>> classify("aspirin", "pain relief medication", api_key="your_key")
        "MEDICAL"
        
        >>> classify("aspirin", "pain relief medication", api_key="your_key", 
        ...          clean_output=True, allowed_categories=["MEDICAL", "FOOD"])
        ["MEDICAL"]
        
        >>> # Using explanation feature
        >>> custom_prompt = "Explain why, then say EXPLANATION_COMPLETE, then classify: ..."
        >>> classify("aspirin", "pain relief", api_key="your_key", baseprompt=custom_prompt,
        ...          clean_output=True, explanation=True, 
        ...          allowed_categories=["MEDICAL", "FOOD"])
        ["MEDICAL"]
        
        >>> # Getting both classification and explanation
        >>> categories, explanation = classify("aspirin", "pain relief", api_key="your_key", 
        ...                                     baseprompt=custom_prompt, clean_output=True,
        ...                                     explanation=True, output_explanation=True,
        ...                                     allowed_categories=["MEDICAL", "FOOD"])
        >>> print(categories)  # ["MEDICAL"]
        >>> print(explanation)  # "Aspirin is widely used as a pain reliever..."
    """
    
    if custom_client is not None:
        client = custom_client
    elif model == "deepseek-chat":
        client = OpenAI(
                        api_key=api_key,
                        base_url="https://api.deepseek.com"
                        )
    else:
        client = OpenAI(
                        api_key=api_key
                        )

    if clean_output and allowed_categories is None:
        raise ValueError("If clean_output is True, a list in allowed_categories must be provided to filter the output.")
    
    if output_explanation and not explanation:
        raise ValueError("If output_explanation is True, explanation must also be True.")
    
    split_base = baseprompt.split("COMPOUND_NAME")
    prompt = split_base[0] + str(name) + split_base[1] + str(input_text)
    prompt = prompt[:max_length]

    
    response = client.responses.create(
        model="gpt-4.1",
        tools=[{ "type": "web_search", "external_web_access": True}],
        tool_choice="required",
        input=prompt,
        )

    if not clean_output:
        return response.output_text
    else:
        cleaned_response_string = response.output_text.replace("\n", " ").replace("  ", " ").strip()

        if explanation:
            # Split by separator and extract classification part
            parts = cleaned_response_string.split(explanation_separator)
            if len(parts) < 2:
                raise ValueError(
                    f"Explanation separator '{explanation_separator}' not found in model response. "
                    f"When explanation=True, the model must include the separator in its response. "
                    f"Response received: {cleaned_response_string[:200]}..."
                )
            # Take everything after the first occurrence of the separator
            cleaned_response_string = parts[1].strip()
            cleaned_explanation = parts[0].strip()
        
        classification_list = cleaned_response_string.split(",")
        classification_list = [item.strip().replace("  ", " ") for item in classification_list]
        
        if allowed_categories is not None:
            updated_classification_list = []
            for item in classification_list:
                if spell_checker is not None:
                    updated_item = spell_checker.correction(item)
                    if updated_item in allowed_categories:
                        updated_classification_list.append(updated_item)
                else:
                    # Fallback to original item if no spell checker provided
                    if item.upper() in [cat.upper() for cat in allowed_categories]:
                        updated_classification_list.append(item)
            
            if explanation and output_explanation:
                return updated_classification_list, cleaned_explanation
            else:
                return updated_classification_list
        else:
            if explanation and output_explanation:
                return classification_list, cleaned_explanation
            else:
                return classification_list





: 

In [ ]:


prompt_search_gpt = "You are a helpful scientist that will classify the provided compound \
COMPOUND_NAME using only the information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"


async def async_run_search(data_list):
    # Get the current running event loop instead of get_event_loop()
    loop = asyncio.get_running_loop()
    # Create tasks for each row - now passing tuples of (name, text)
    futures = [loop.run_in_executor(None, custom_search_classify, name, text, openai_api_key, prompt_search_gpt) for name, text in data_list]
    # Gather all results
    result = await asyncio.gather(*futures, return_exceptions=True)
    return result


: 

In [ ]:
start_time_search_gpt = time.perf_counter()

classifications_search_gpt = await async_run_search(drug_library_text_random_sample_data)

end_time_search_gpt = time.perf_counter()

: 

In [ ]:
time_elapsed_search_gpt = end_time_search_gpt - start_time_search_gpt
print(f"Time elapsed for 100 compounds (seconds): {time_elapsed_search_gpt}")

total_time_estimate_search_gpt = time_elapsed_search_gpt * token_ratio
print(f"Total time estimate (seconds, Search GPT): {total_time_estimate_search_gpt}")

Time elapsed for 100 compounds (seconds): 26.595775332985795
Total time estimate (seconds, Search GPT): 1190.2036241088201


: 

In [ ]:
classifications_search_gpt.append([time_elapsed_search_gpt])

: 

In [ ]:
pd.DataFrame(classifications_search_gpt).to_csv("../data/drug_library/measure_time_sample_results/search_gpt_time_classification_results.csv")

: 

In [ ]:
start_time_search_gpt_linear = time.perf_counter()

classifications_search_gpt_linear = []
for index, row in drug_library_text_random_sample.iterrows():
    compound_name = row["NAME"]
    compound_text = row["TEXT"]
    classification = custom_search_classify(compound_name, compound_text, openai_api_key, prompt_search_gpt)
    classifications_search_gpt_linear.append(classification)

end_time_search_gpt_linear = time.perf_counter()

: 

In [ ]:
time_elapsed_search_gpt_linear = end_time_search_gpt_linear - start_time_search_gpt_linear
print(f"Time elapsed for 100 compounds (seconds): {time_elapsed_search_gpt_linear}")

total_time_estimate_search_gpt_linear= time_elapsed_search_gpt_linear * token_ratio
print(f"Total time estimate (seconds, Search GPT, linear): {total_time_estimate_search_gpt_linear}")

Time elapsed for 100 compounds (seconds): 290.1188310839934
Total time estimate (seconds, Search GPT, linear): 12983.283241610168


: 

In [ ]:
with open("../data/drug_library/measure_time_sample_results/search_gpt_linear_time_results.txt", "a") as f:
    f.write(str(time_elapsed_search_gpt_linear))

: 

In [ ]:
deepinfra_api_key = open("../secrets/deepinfra_api_key.txt").read().strip()


deepseek_client = OpenAI(
    api_key=deepinfra_api_key,
    base_url="https://api.deepinfra.com/v1/openai",
)

chem_deepseek = chemsource.ChemSource(
    model_api_key=openai_api_key,
    ncbi_key=ncbi_api_key,
    model="deepseek-ai/DeepSeek-V3.1",
    clean_output=True,
    allowed_categories=["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"],
    custom_client=deepseek_client
)

prompt_deepseek = "You are a helpful scientist that will classify the provided compound \
COMPOUND_NAME using only the information provided as any combination of the \
following: MEDICAL, ENDOGENOUS, FOOD, PERSONAL CARE, INDUSTRIAL. Note that \
MEDICAL refers to compounds actively used as approved medications in \
humans or in late-stage clinical trials in humans. Note that ENDOGENOUS \
refers to compounds that are produced by the human body specifically. \
ENDOGENOUS excludes essential nutrients that cannot be synthesized by the \
human body. Note that FOOD refers to compounds present in natural food items \
or food additives. Note that PERSONAL CARE refers to non-medicated compounds \
typically used for activities such as skincare, beauty, and fitness. Note \
that INDUSTRIAL should be used only for synthetic compounds not used as a \
contributing ingredient in the medical, personal care, or food industries. \
Specify INFO instead if more information is needed. DO NOT MAKE ANY \
ASSUMPTIONS, USE ONLY THE INFORMATION PROVIDED AFTER THE COMPOUND NAME \
BY THE USER. A classification of INFO will also be rewarded when \
correctly applied and is strongly encouraged if information is of poor \
quality, if there is not enough information, or if you are not completely \
confident in your answer.  Provide the output as a plain text separated \
by commas, and provide only the categories listed (either list a \
combination of INDUSTRIAL, ENDOGENOUS, PERSONAL CARE, MEDICAL, FOOD or \
list INFO), with no justification. Provided Information:\n"


chem_deepseek.prompt = prompt_deepseek

: 

In [ ]:
start_time_deepseek = time.perf_counter()

classifications_deepseek = await async_run(drug_library_text_random_sample_data, chem_deepseek)

end_time_deepseek = time.perf_counter()

: 

In [ ]:
time_elapsed_deepseek = end_time_deepseek - start_time_deepseek
print(f"Time elapsed for 100 compounds (seconds): {time_elapsed_deepseek}")

total_time_estimate_deepseek = time_elapsed_deepseek * token_ratio
print(f"Total time estimate (seconds, DeepSeek): {total_time_estimate_deepseek}")

Time elapsed for 100 compounds (seconds): 8.564955291003571
Total time estimate (seconds, DeepSeek): 383.2954933650367


: 

In [ ]:
classifications_deepseek.append([time_elapsed_deepseek])

: 

In [ ]:
pd.DataFrame(classifications_deepseek).to_csv("../data/drug_library/measure_time_sample_results/deepseek_time_classification_results.csv")

: 

In [ ]:
start_time_deepseek_linear = time.perf_counter()

classifications_deepseek_linear = []
for index, row in drug_library_text_random_sample.iterrows():
    compound_name = row["NAME"]
    compound_text = row["TEXT"]
    classification = chem_deepseek.classify(compound_name, compound_text)
    classifications_deepseek_linear.append(classification)

end_time_deepseek_linear = time.perf_counter()

: 

In [ ]:
time_elapsed_deepseek_linear = end_time_deepseek_linear - start_time_deepseek_linear
print(f"Time elapsed for 100 compounds (seconds): {time_elapsed_deepseek_linear}")

total_time_estimate_deepseek_linear= time_elapsed_deepseek_linear * token_ratio
print(f"Total time estimate (seconds, DeepSeek, linear): {total_time_estimate_deepseek_linear}")

Time elapsed for 100 compounds (seconds): 71.16816045800806
Total time estimate (seconds, DeepSeek, linear): 3184.8893832857398


: 

In [ ]:
with open("../data/drug_library/measure_time_sample_results/deepseek_linear_time_results.txt", "a") as f:
    f.write(str(time_elapsed_deepseek_linear))

: 